# 4 · Many stations, one chain each

Many locations at once: one state per station (or grid cell).

This is how the chain runs in production for a whole country: a mapping from
location to `FwiState`, advanced once a day. Each location is independent,
so the loop parallelises trivially (processes, a task queue, a SQL job...).

The same code as [`examples/04_many_stations.py`](../examples/04_many_stations.py).

> **A fire weather index is not an alert.** It says how easily a fire would start and spread in the weather given, not that one will. The official fire danger for Europe is published by [EFFIS](https://forest-fire.emergency.copernicus.eu/).

In [1]:
import sys
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path("../examples").resolve()))  # weather.py, the synthetic season
sys.argv = ["notebook", str(Path(tempfile.mkdtemp()) / "unused")]

In [2]:
from weather import synthetic_season

from fwi_chain import STANDARD_PARAMS, FwiState, advance

# Three toy "stations": the same synthetic climate with a different seed,
# i.e. different weather on each day.
stations = {"north-valley": 11, "coastal-plain": 7, "inland-hills": 3}
weather = {name: list(synthetic_season(seed=seed)) for name, seed in stations.items()}
states: dict[str, FwiState] = dict.fromkeys(stations, STANDARD_PARAMS.initial_state)

n_days = len(next(iter(weather.values())))
for i in range(n_days):
    fwi_today = {}
    for name in stations:
        r = weather[name][i]
        out = advance(
            states[name],
            month=r.day.month,
            temperature_c=r.temperature_c,
            relative_humidity_pct=r.relative_humidity_pct,
            wind_speed_kmh=r.wind_speed_kmh,
            rain_24h_mm=r.rain_24h_mm,
            params=STANDARD_PARAMS,
        )
        states[name] = out.state
        fwi_today[name] = out.fwi
    day = weather["north-valley"][i].day
    if day.day == 1:  # print the first of each month
        cells = "  ".join(f"{name} {fwi:5.1f}" for name, fwi in fwi_today.items())
        print(f"{day}  {cells}")

2026-04-01  north-valley   3.9  coastal-plain   2.1  inland-hills   4.7
2026-05-01  north-valley  18.1  coastal-plain  20.4  inland-hills  17.6
2026-06-01  north-valley  25.6  coastal-plain   2.3  inland-hills  38.9
2026-07-01  north-valley  46.1  coastal-plain  69.0  inland-hills  50.9
2026-08-01  north-valley  30.6  coastal-plain  46.6  inland-hills  50.1
2026-09-01  north-valley  34.5  coastal-plain  38.2  inland-hills  24.9
2026-10-01  north-valley  34.2  coastal-plain   5.9  inland-hills  32.5
